In [ ]:
from modelo import VotoInvalidoException, PistaInvalidaException, ConfiguracionPartida, Partida, Ganador, ValidadorPalabraProhibida, ValidadorLongitud, Jugador
class ControladorVotacion:
    def __init__(self, vista, partida):
        self._vista = vista
        self._partida = partida
    def ejecutar(self):
        activos = self._partida.jugadores_activos
        nombres = [j.nombre for j in activos]
        for votante in activos:
            self._vista.limpiar_pantalla()
            self._vista.esperar_enter(f"Pasa el computador a {votante.nombre}. Enter para votar...")
            while True:
                posicion = self._vista.elegir_opcion(f"{votante.nombre}, ¿quién crees que es el impostor?", nombres)
                try:
                    self._partida.votar(votante, activos[posicion])
                    break
                except VotoInvalidoException as error:
                    self._vista.mostrar_error(str(error))
        self._vista.limpiar_pantalla()
        acusado = self._partida.cerrar_votacion()
        if acusado is None:
            self._vista.mostrar_mensaje("Hubo empate: nadie fue eliminado.")
        else:
            self._vista.mostrar_mensaje(f"{acusado.nombre} fue el más votado y queda eliminado.")
        return acusado
class ControladorRonda:
    def __init__(self, vista, partida, controlador_votacion):
        self._vista = vista
        self._partida = partida
        self._controlador_votacion = controlador_votacion

    def ejecutar(self):
        ronda = self._partida.ronda_actual
        self._vista.mostrar_mensaje(f"\n===== RONDA {ronda.numero} =====")

        for jugador in self._partida.jugadores_activos:
            if ronda.pistas:
                self._vista.mostrar_pistas(ronda.pistas)
            while True:
                texto = self._vista.pedir_texto(f"{jugador.nombre}, escribe tu pista:")
                try:
                    self._partida.dar_pista(jugador, texto)
                    break
                except PistaInvalidaException as error:
                    self._vista.mostrar_error(str(error))
        self._partida.avanzar_fase()
        self._vista.mostrar_pistas(ronda.pistas)
        self._vista.mostrar_mensaje("\nDebatan quién creen que es el impostor.")
        self._vista.esperar_enter("Presionen Enter cuando termine el debate...")

        self._partida.avanzar_fase()
        eliminado = self._controlador_votacion.ejecutar()

        self._partida.avanzar_fase()
        return eliminado
class ControladorPartida:
    def __init__(self, vista, repositorio, asignador, contador_votos):
        self._vista = vista
        self._repositorio = repositorio
        self._asignador = asignador
        self._contador_votos = contador_votos

    def _registrar_jugadores(self):
        opciones = [str(n) for n in range(3, 11)]
        cantidad = self._vista.elegir_opcion("¿Cuántos jugadores van a jugar?", opciones) + 3
        jugadores = []
        usados = set()
        for i in range(1, cantidad + 1):
            while True:
                nombre = self._vista.pedir_texto(f"Nombre del jugador {i}:")
                if not nombre:
                    self._vista.mostrar_error("El nombre no puede estar vacío.")
                elif nombre.lower() in usados:
                    self._vista.mostrar_error("Ese nombre ya está en uso.")
                else:
                    break
            usados.add(nombre.lower())
            jugadores.append(Jugador(i, nombre))
        return jugadores

    def _revelar_roles(self, partida):
        for jugador in partida.jugadores:
            self._vista.limpiar_pantalla()
            self._vista.esperar_enter(f"Pasa el computador a {jugador.nombre}. Enter para ver tu rol...")
            self._vista.mostrar_rol_privado(jugador)
            self._vista.esperar_enter("Enter para ocultarlo y pasar al siguiente...")
        self._vista.limpiar_pantalla()

    def jugar(self):
        self._vista.mostrar_mensaje("EL IMPOSTOR")
        jugadores = self._registrar_jugadores()

        categorias = self._repositorio.listar_categorias()
        posicion = self._vista.elegir_opcion("Elige la categoría:", categorias)
        categoria = categorias[posicion]

        configuracion = ConfiguracionPartida(3, 60, categoria, True, 20)
        palabra = self._repositorio.obtener_palabra_aleatoria(categoria)
        validadores = [ValidadorPalabraProhibida(palabra),
                       ValidadorLongitud(configuracion.longitud_maxima_pista)]
        partida = Partida(jugadores, palabra, configuracion,
                          self._asignador, self._contador_votos, validadores)
        partida.iniciar()
        self._revelar_roles(partida)

        controlador_votacion = ControladorVotacion(self._vista, partida)
        controlador_ronda = ControladorRonda(self._vista, partida, controlador_votacion)
        while True:
            controlador_ronda.ejecutar()
            resultado = partida.determinar_ganador()
            if (resultado is not None and resultado.ganador == Ganador.CIVILES
                    and configuracion.ultima_oportunidad):
                intento = self._vista.pedir_texto(
                    f"{resultado.impostor.nombre}, te descubrieron. Última oportunidad, intenta adivinar la palabra:")
                resultado = partida.determinar_ganador(intento)
            if resultado is not None:
                break
            partida.nueva_ronda()
        self._vista.mostrar_resultado(resultado)
        return resultado